# 0. Imports & Reproducibility

In [ ]:
import random
from collections import defaultdict
from pathlib import Path

import community as community_louvain
import faiss
import networkx as nx
import numpy as np
import pandas as pd
import polars as pl
import torch
from gensim.models import Word2Vec
from node2vec import Node2Vec
from sentence_transformers import SentenceTransformer


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 1. Download and Store Dataset

In [3]:
def ingest_reddit_data(
    subreddit_key: str, n_rows: int = 1_000_000, force_rerun: bool = False
) -> Path:
    """
    Orchestrates the ETL process for a specific subreddit's comment data.

    Args:
        subreddit_key: Dictionary key from 'splits' (e.g., 'changemyview').
        n_rows: Maximum records to process for the local sample.
        force_rerun: If True, bypasses existence check and overwrites existing parquet file.

    Returns:
        Path to the processed Parquet file.
    """
    out_path = Path(f"data/processed/{subreddit_key}_sample.parquet")

    # Idempotency check: Skip heavy network I/O if the target file is already present
    if out_path.exists() and not force_rerun:
        print(f"Skipping ingestion: Local cache found at {out_path}")
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Define schema subset based on downstream analytical requirements
    feature_cols = [
        "author",
        "body",
        "created_utc",
        "id",
        "link_id",
        "name",
        "parent_id",
        "score",
        "controversiality",
        "total_awards_received",
    ]
    splits = {
        "changemyview": "data/changemyview-*-of-*.parquet",
    }

    print(f"Streaming data from HuggingFace for: r/{subreddit_key}...")

    # Execute lazy-evaluated ETL pipeline
    try:
        (
            pl.scan_parquet(
                f"hf://datasets/HuggingFaceGECLM/REDDIT_comments/{splits[subreddit_key]}"
            )
            .select(feature_cols)
            # Filter out deleted/removed content to maintain high data quality for NLP tasks
            .filter(~pl.col("body").is_in(["[deleted]", "[removed]"]))
            .limit(n_rows)
            # Stream directly to disk using ZSTD to balance compression ratio and write speed
            .sink_parquet(out_path, compression="zstd")
        )
        print(f"Successfully wrote {n_rows} rows to {out_path}")
    except KeyError:
        raise ValueError(f"Subreddit '{subreddit_key}' not found in defined splits.")
    except Exception as e:
        print(f"Pipeline failed: {e}")
        raise

    return out_path


# --- Execution Control ---
# Toggle 'force_rerun' if the upstream data schema changes or a larger sample is needed
OUT = ingest_reddit_data("changemyview", n_rows=1_000_000, force_rerun=False)

Skipping ingestion: Local cache found at data/processed/changemyview_sample.parquet


# 2. Load Data from Parquet File

In [4]:
df = (
    # Scan the metadata and define the lazy query plan
    pl.scan_parquet("data/processed/changemyview_sample.parquet")
    # Constrain sample size for rapid local prototyping
    .head(30000)
    # Trigger execution and load into memory
    .collect()
    # Bridge to Pandas for ecosystem compatibility
    .to_pandas()
)

# 3. Create Train and Test Dataset

### 3.1 Data Preprocessing, Temporal Splitting & Metadata Mapping

In [5]:
# --- 1. Data Cleaning & Type Casting ---

# Ensure text integrity by removing null observations in the primary feature
df = df.dropna(subset=["body"])

# Filter out anonymous/deleted accounts to maintain attribution quality
df = df[df["author"] != "[deleted]"]

# Normalize timestamps: Convert raw strings to numeric Unix seconds, then to datetime objects
# 'coerce' handles malformed strings by returning NaT, preventing pipeline crashes
df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")
df["date"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")


# --- 2. Temporal Train/Test Split ---

# Use a temporal 80/20 split rather than a random shuffle to prevent 'look-ahead' bias.
# This simulates a real-world scenario where we predict future comments based on past data.
cutoff = df["created_utc"].quantile(0.8)

df_train = df[df["created_utc"] <= cutoff].copy()
df_test = df[df["created_utc"] > cutoff].copy()


# --- 3. Metadata Mapping (Lookup Tables) ---

# Create lightweight author lookups for efficient O(1) retrieval.
# Mappings are scoped strictly within splits to enforce isolation and prevent leakage.
id2author_train = df_train.set_index("id")["author"].to_dict()
id2author_test = df_test.set_index("id")["author"].to_dict()

### 3.2 Interaction Network Construction

In [6]:
def build_reply_pairs(df_split, id2author):
    """
    Constructs a positive interaction dataset by mapping comments to their parent authors.
    Filters for comment-to-comment replies and removes self-interactions.
    """
    # Reddit 'parent_id' prefixes: t1 = Comment, t3 = Link/Post.
    # We restrict analysis to comment-to-comment interactions to capture conversational dynamics.
    parent_comment_ids = df_split["parent_id"].astype(str)
    is_comment_reply = parent_comment_ids.str.startswith("t1_")
    df_r = df_split[is_comment_reply].copy()

    # Extract the raw 36-base ID by stripping the 't1_' type prefix for join compatibility
    df_r["parent_key"] = df_r["parent_id"].str.replace("^t1_", "", regex=True)

    # Resolve parent author identities via the provided lookup table (O(1) mapping)
    df_r["parent_author"] = df_r["parent_key"].map(id2author)

    # --- Data Integrity & Quality Filtering ---
    # 1. Drop replies where the parent comment falls outside the current split (boundary integrity)
    df_r = df_r.dropna(subset=["parent_author"])
    # 2. Exclude self-replies to ensure we only model interpersonal interactions
    df_r = df_r[df_r["author"] != df_r["parent_author"]]

    # Feature selection and renaming to standard (u, v) graph notation
    pairs_pos = df_r[
        ["author", "parent_author", "created_utc", "link_id", "id", "parent_key"]
    ].copy()
    pairs_pos = pairs_pos.rename(
        columns={
            "author": "u",
            "parent_author": "v",
            "id": "u_comment_id",
            "parent_key": "v_comment_id",
        }
    )

    # Label as positive instances for downstream binary classification
    pairs_pos["y"] = 1
    return pairs_pos


# Generate interaction sets; scoped within splits to prevent data leakage
pos_train = build_reply_pairs(df_train, id2author_train)
pos_test = build_reply_pairs(df_test, id2author_test)

print(f"Positive samples - Train: {len(pos_train):,} | Test: {len(pos_test):,}")

Positive samples - Train: 12,808 | Test: 3,114


In [7]:
# --- Graph Diagnostics: Sparsity & Degree Distribution ---

# Calculate the ratio of users who engaged in at least one reply
pos_users = set(pos_train["u"]) | set(pos_train["v"])
all_users = set(df_train["author"].dropna().unique())
print(
    f"Engagement Coverage: {len(pos_users)} / {len(all_users)} users with interactions"
)

# Analyze the 'Out-Degree' (number of replies sent per user)
print("\nReplies per user statistics:")
print(pos_train.groupby("u").size().describe())

Engagement Coverage: 2502 / 3265 users with interactions

Replies per user statistics:
count    2249.000000
mean        5.694976
std        16.164946
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       467.000000
dtype: float64


### 3.3 Negative Sampling Strategy

In [8]:
def build_hard_negatives(df_split, pos_pairs, k_per_pos=2, seed=42):
    """
    Generates 'hard' negative samples for link prediction by identifying potential
    interactions that did NOT occur within the same discussion thread context.
    """
    # Initialize a BitGenerator for reproducible stochastic sampling
    rng = np.random.default_rng(seed)

    # 1) Contextual Mapping: Identify all active participants per discussion thread (link_id).
    # This defines our 'closed-world' candidate pool for each observation.
    thread_users = (
        df_split.groupby("link_id")["author"].apply(lambda s: set(s.dropna())).to_dict()
    )

    # 2) Network Topology: Extract existing interaction edges in (u, v) space.
    # We treat edges as symmetric to prevent sampling reciprocal replies as negatives,
    # which would introduce label noise.
    reply_edges = set(zip(pos_pairs["u"], pos_pairs["v"]))
    reply_edges_sym = reply_edges | {(v, u) for (u, v) in reply_edges}

    neg_rows = []
    # Project to minimal feature set to reduce overhead during iteration
    pos_pairs_small = pos_pairs[["u", "v", "link_id"]].copy()

    for u, v, link_id in pos_pairs_small.itertuples(index=False):
        users = list(thread_users.get(link_id, []))
        if len(users) <= 1:
            continue

        # Candidate Filtering:
        # Target users in the same thread (high-signal 'hard' negatives) excluding the source 'u'
        cand = [x for x in users if x != u]
        if not cand:
            continue

        # Collision Avoidance: Remove candidates where a ground-truth interaction (u, x) exists
        cand = [x for x in cand if (u, x) not in reply_edges_sym]
        if not cand:
            continue

        # Stochastic Sampling: Select 'k' negatives per positive to maintain class ratio
        take = min(k_per_pos, len(cand))
        sampled = rng.choice(cand, size=take, replace=False)

        for x in sampled:
            neg_rows.append((u, x, link_id, 0))

    return pd.DataFrame(neg_rows, columns=["u", "v", "link_id", "y"])


# --- Triplet Dataset Assembly ---


def build_triplets_from_hard_negatives(pos_pairs, neg_pairs, seed=42):
    """
    Constructs triplets (u, v_pos, v_neg) for metric learning.
    For each positive interaction (u, v_pos), sample one hard negative v_neg
    from the same thread context.
    """
    rng = np.random.default_rng(seed)

    # Map u → list of negative candidates
    neg_map = neg_pairs.groupby("u")["v"].apply(list).to_dict()

    triplets = []

    for u, v_pos, link_id in pos_pairs[["u", "v", "link_id"]].itertuples(index=False):
        neg_candidates = neg_map.get(u, [])
        if not neg_candidates:
            continue

        # Sample one hard negative for this positive
        v_neg = rng.choice(neg_candidates)

        triplets.append((u, v_pos, v_neg, link_id))

    return pd.DataFrame(triplets, columns=["u", "v_pos", "v_neg", "link_id"])


# --- Generate hard negatives (unchanged) ---
neg_train = build_hard_negatives(df_train, pos_train, k_per_pos=2)
neg_test = build_hard_negatives(df_test, pos_test, k_per_pos=2)

# --- Build triplets ---
triplets_train = build_triplets_from_hard_negatives(pos_train, neg_train)
triplets_test = build_triplets_from_hard_negatives(pos_test, neg_test)

print("Triplets Train:", len(triplets_train))
print("Triplets Test:", len(triplets_test))
print(triplets_train.head())


Triplets Train: 12747
Triplets Test: 2944
                 u                 v_pos            v_neg    link_id
0        Jaberkaty  Thompson_S_Sweetback  ancillarynipple  t3_16ralh
1  ancillarynipple  Thompson_S_Sweetback        Jaberkaty  t3_16ralh
2  ancillarynipple  Thompson_S_Sweetback        Jaberkaty  t3_16ralh
3           llatia             gchase723          banebot  t3_16s6jg
4     cardswsbound                 nix0n     mavriksfan11  t3_16rzx1


In [9]:
def test_triplet_integrity(triplets, pos_df, neg_df):
    """
    Comprehensive suite to verify triplet logic and data leakage.
    """
    # 1. Structural Check
    assert not triplets.isnull().values.any(), "Triplets contain NaN values"

    # 2. Contextual Integrity: v_neg must actually exist in the negative pool for that user
    # This ensures rng.choice didn't pull a random user from the wrong context
    u_to_negs = neg_df.groupby("u")["v"].apply(set).to_dict()

    for row in triplets.itertuples():
        # Check: v_pos and v_neg must be different
        assert row.v_pos != row.v_neg, f"Anchor {row.u} has identical Pos/Neg target"

        # Check: anchor cannot be its own target
        assert row.u != row.v_pos, f"Self-loop found in positive: {row.u}"
        assert row.u != row.v_neg, f"Self-loop found in negative: {row.u}"

        # Check: v_neg must be a valid 'hard' negative from the pool
        valid_negs = u_to_negs.get(row.u, set())
        assert row.v_neg in valid_negs, (
            f"User {row.v_neg} is not a valid hard negative for {row.u}"
        )

    # 3. Label Leakage: Ensure v_neg is NEVER a real positive for that user
    # (Symmetric check to be extra safe)
    pos_edges = set(zip(pos_df["u"], pos_df["v"]))
    pos_edges_sym = pos_edges | {(v, u) for (u, v) in pos_edges}

    triplet_neg_edges = set(zip(triplets["u"], triplets["v_neg"]))
    overlap = triplet_neg_edges.intersection(pos_edges_sym)

    assert len(overlap) == 0, (
        f"Leakage detected! {len(overlap)} 'negatives' are actually real interactions."
    )

    print("✅ All Triplet Integrity Tests Passed!")


# Run the test
test_triplet_integrity(triplets_train, pos_train, neg_train)

✅ All Triplet Integrity Tests Passed!


### 3.4 User Textual Profile Construction

In [10]:
# --- 1. Corpus Preparation & Leakage Prevention ---

# Isolate training and testing text to ensure that future comments do not
# influence the historical representations of users in the training set.
df_train_text = df_train.dropna(subset=["body", "id"]).copy()
df_test_text = df_test.dropna(subset=["body", "id"]).copy()

# (Optional) Heuristic: Filter for active users to ensure embeddings have
# sufficient signal (min 5 observations).
# df_train_text = df_train_text.groupby("id").filter(lambda g: len(g) >= 5)

# --- 2. Temporal Aggregation (Feature Engineering) ---

# Construct a profile for each author.
# We join the most recent comments to capture the user's current interests/voice.
user_text_train = (
    df_train_text.sort_values(
        "created_utc"
    )  # Enforce chronology to correctly identify the 'tail'
    .groupby("author")["body"]
    # Hyperparameter: Concatenating the last 10 comments balances context vs. sequence length
    .apply(lambda s: " ".join(s.tail(10)))
)

user_text_test = (
    df_test_text.sort_values("created_utc")
    .groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
)

# Convert to hash maps (dict) for O(1) lookup performance during the mapping phase
user_text_dict_train = user_text_train.to_dict()
user_text_dict_test = user_text_test.to_dict()

# --- 3. Coverage Analysis (Data Integrity Check) ---

# Quantify the 'Cold-Start' issue: users in the interaction pairs who lack
# textual history. Significant missingness here indicates a sampling mismatch.
missing_train = triplets_train["u"].map(user_text_dict_train).isna().mean()
print(f"Missing text profile ratio (Train - Source User): {missing_train:.2%}")

missing_test = triplets_test["u"].map(user_text_dict_test).isna().mean()
print(f"Missing text profile ratio (Test - Source User): {missing_test:.2%}")

Missing text profile ratio (Train - Source User): 0.00%
Missing text profile ratio (Test - Source User): 0.00%


In [11]:
# --- 1. Prepare Data Containers ---

# Create copies to prevent SettingWithCopy warnings and isolate split changes
triplets_train = triplets_train.copy()
triplets_test = triplets_test.copy()


def attach_text(triplets, user_text_dict):
    """Adds historical text for source (u) and target (v) users."""

    # Map text profiles to user IDs
    triplets["text_u"] = triplets["u"].map(user_text_dict)
    triplets["text_v_pos"] = triplets["v_pos"].map(user_text_dict)
    triplets["text_v_neg"] = triplets["v_neg"].map(user_text_dict)

    # Remove observations missing text for either user to ensure a complete feature set
    return triplets.dropna(subset=["text_u", "text_v_pos", "text_v_neg"])


# --- 2. Execute Merge & Cleanup ---

triplets_train_txt = attach_text(triplets_train, user_text_dict_train)
triplets_test_txt = attach_text(triplets_test, user_text_dict_test)

# --- 3. Progress Check ---

# Log row counts to monitor data loss during the mapping/dropping process
print(f"Train Retention: {len(triplets_train):,} -> {len(triplets_train_txt):,}")
print(f"Test Retention:  {len(triplets_test):,} -> {len(triplets_test_txt):,}")

Train Retention: 12,747 -> 12,747
Test Retention:  2,944 -> 2,944


### 3.5 Save Train and Test Datasets

In [12]:
triplets_train_txt.to_parquet("data/processed/train_triplets_txt.parquet", index=False)
triplets_test_txt.to_parquet("data/processed/test_triplets_txt.parquet", index=False)

# 4.  Implement Baselines

### 4.1 Random Baseline

In [13]:
# 1. Candidates = all users from training
all_users_train = set(pos_train["u"]).union(set(pos_train["v"]))
all_users_train = list(all_users_train)

def recommend_random(user, k=10, exclude_seen=True):
    """
    Recommend k random users.
    
    Args:
        user: source user
        k: number of recommendations
        exclude_seen: avoid recommending already interacted users (train)
    """
    if not exclude_seen:
        candidates = [u for u in all_users_train if u != user]
        return random.sample(candidates, k)

    # Users already interacted with in training
    seen = set(pos_train[pos_train["u"] == user]["v"].values)

    candidates = [
        u for u in all_users_train
        if u != user and u not in seen
    ]

    if len(candidates) < k:
        return candidates

    return random.sample(candidates, k)

### 4.2 Common Neighbor Baseline

In [14]:
# 1. Build an Adjacency List from the TRAIN set
# We treat the network as undirected to find "friends of friends" (mutual interactors)
adj = defaultdict(set)
for u, v in zip(pos_train["u"], pos_train["v"]):
    adj[u].add(v)
    adj[v].add(u)

def recommend_common_neighbors(user, k=10, exclude_seen=True):
    """
    Recommend users based on the number of shared interaction partners (Common Neighbors).
    
    Args:
        user: The source user for whom to generate recommendations.
        k: Number of recommendations to return.
        exclude_seen: If True, prevents recommending users already interacted with in training.
    """
    if user not in adj:
        # Cold-start: If the user has no history, no neighbors can be found
        return []

    user_neighbors = adj[user]
    candidate_scores = defaultdict(int)
    
    # Traverse to neighbors (friends) and then to their neighbors (friends of friends)
    for neighbor in user_neighbors:
        for fof in adj[neighbor]: # Friend of a Friend
            if fof != user:
                # Increment score for every shared path (common neighbor)
                candidate_scores[fof] += 1
    
    # Filter: Remove users the target user has already interacted with in the training set
    if exclude_seen:
        for seen_user in user_neighbors:
            if seen_user in candidate_scores:
                del candidate_scores[seen_user]
                
    # Sort candidates by the number of common neighbors in descending order
    sorted_recs = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    return [rec[0] for rec in sorted_recs[:k]]

### 4.3 Popularity based Baseline

In [15]:
# 1. Compute popularity scores from TRAIN only
popularity_scores = (
    pos_train["v"]
    .value_counts()
    .to_dict()
)

# 2. Global ranking of users by popularity
global_pop_ranking = sorted(
    popularity_scores.keys(),
    key=lambda x: popularity_scores[x],
    reverse=True
)


def recommend_popularity(user, k=10, exclude_seen=True):
    """
    Recommend top-k most popular users.
    
    Args:
        user: source user
        k: number of recommendations
        exclude_seen: avoid recommending already interacted users (train)
    """
    if not exclude_seen:
        return global_pop_ranking[:k]

    # Users already interacted with in training
    seen = set(pos_train[pos_train["u"] == user]["v"].values)

    recs = []
    for candidate in global_pop_ranking:
        if candidate != user and candidate not in seen:
            recs.append(candidate)
        if len(recs) == k:
            break

    return recs

# 5. Create Text Embeddings

In [ ]:
# Load pretrained model
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

def generate_transformer_embeddings(unique_user_df, batch_size=64):
    user_ids = unique_user_df["author"].tolist()
    texts = unique_user_df["body"].tolist()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return {
        u: vec for u, vec in zip(user_ids, embeddings)
    }

unique_users = (
    df_train.groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
    .reset_index()
)

# Generate embeddings
text_embeddings = generate_transformer_embeddings(unique_users)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/52 [00:00<?, ?it/s]

# 6. Create Graph Embeddings

In [17]:
# ======================================================
# 5.X Train Node2Vec on directed training graph
# ======================================================

user_id_list = unique_users["author"].tolist()


# Build directed graph from training interactions
G = nx.DiGraph()

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    G.add_edge(u, v)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

# Train Node2Vec
node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=30,
    num_walks=200,
    workers=1,
    p=1.0,
    q=1.0,
    seed=42
)


n2v_model = Word2Vec(
    node2vec.walks, 
    vector_size=64, 
    window=10, 
    min_count=1, 
    batch_words=128,
    seed=42, 
    workers=1
)

# Extract graph embeddings aligned with text embeddings
graph_dim = 64
graph_embeddings = {}

for node in G.nodes():
    vec = n2v_model.wv[str(node)]
    
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
        
    graph_embeddings[node] = vec

Graph nodes: 2502
Graph edges: 7921


Computing transition probabilities:   0%|          | 0/2502 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 200/200 [00:27<00:00,  7.39it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


# 6. Set up a Vector Search Index

In [18]:
# ======================================================
# Build FAISS index (Graph + Text Combined)
# ======================================================

joint_embeddings = {}
joint_vectors = []

# Only keep users that exist in BOTH embeddings
common_users = [
    u for u in graph_embeddings
    if u in text_embeddings
]

for u in common_users:
    g = graph_embeddings[u]
    t = text_embeddings[u]

    # Normalize each modality separately
    g = g / (np.linalg.norm(g) + 1e-9)
    t = t / (np.linalg.norm(t) + 1e-9)

    # Concatenate
    z = np.concatenate([g, t])

    # Final normalization
    z = z / (np.linalg.norm(z) + 1e-9)

    joint_embeddings[u] = z
    joint_vectors.append(z)

user_id_list = common_users
joint_matrix = np.vstack(joint_vectors).astype("float32")

# Normalize again for cosine similarity (FAISS requirement)
faiss.normalize_L2(joint_matrix)

d = joint_matrix.shape[1]
index = faiss.IndexFlatIP(d)
index.add(joint_matrix)

user_vectors = {
    u: joint_matrix[i]
    for i, u in enumerate(user_id_list)
}

print("Joint Graph+Text FAISS index built with", index.ntotal, "users.")

Joint Graph+Text FAISS index built with 2502 users.


# 7. Implement the Retrieval Function

In [19]:
def build_candidate_pool(u):
    candidates = set()

    # 1️⃣ Same thread participants (strongest signal)
    threads_u = set(pos_train[pos_train["u"] == u]["link_id"])
    for t in threads_u:
        users_in_thread = set(
            pos_train[pos_train["link_id"] == t]["u"]
        )
        candidates.update(users_in_thread)

    # 2️⃣ 2-hop neighbors
    if u in G:
        one_hop = set(G.neighbors(u))
        for n in one_hop:
            candidates.update(G.neighbors(n))
            list(G.neighbors(n))

    # Remove self and existing connections
    existing = set(pos_train[pos_train["u"] == u]["v"])
    candidates -= existing
    candidates.discard(u)

    return list(candidates)

In [20]:
test_user = user_id_list[1]

candidates = build_candidate_pool(test_user)

print("User:", test_user)
print("Number of candidates:", len(candidates))
print("First 10 candidates:", candidates[:10])

User: Thompson_S_Sweetback
Number of candidates: 1
First 10 candidates: ['Jaberkaty']


In [21]:
assert test_user not in candidates, "Self recommendation detected!"

In [22]:
existing = set(pos_train[pos_train["u"] == test_user]["v"])

overlap = set(candidates) & existing

print("Overlap with existing edges:", overlap)
assert len(overlap) == 0, "Existing edges not removed!"

Overlap with existing edges: set()


# 8. Implement Score Function

In [23]:
def score(u, v, alpha=0):
    graph_sim = np.dot(graph_embeddings[u], graph_embeddings[v])
    text_sim = np.dot(text_embeddings[u], text_embeddings[v])
    return alpha * graph_sim + (1-alpha) * text_sim

# 9. Implement the Recommendation Function

In [24]:
train_edges = defaultdict(set)

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    train_edges[u].add(v)   # only outgoing

def recommend_cnn(u, k=10, fallback_search_k=200):

    if u not in graph_embeddings:
        return []

    # Existing neighbors (must not be recommended)
    existing = train_edges.get(u, set())

    # ==============================
    # Stage 1: Structured Retrieval
    # ==============================
    candidates = build_candidate_pool(u)

    scored = []

    for v in candidates:
        if v not in graph_embeddings:
            continue
        if v in existing:
            continue

        s = score(u, v)
        scored.append((v, s))

    scored.sort(key=lambda x: x[1], reverse=True)
    recs = [v for v, _ in scored]

    # ==============================
    # Stage 2: FAISS Fallback
    # ==============================
    if len(recs) < k:

        query_vec = joint_embeddings[u].reshape(1, -1).astype("float32")

        distances, indices = index.search(query_vec, fallback_search_k)

        for idx in indices[0]:
            v = user_id_list[idx]

            if v == u:
                continue
            if v in existing:
                continue
            if v in recs:
                continue

            recs.append(v)

            if len(recs) == k:
                break

    return recs[:k]

In [25]:
def test_recommendation(user_id):
    if user_id not in user_vectors:
        return "User not found"

    recs = recommend_cnn(user_id, k=3)

    print(
        f"Target User ({user_id}) history sample: {unique_users[unique_users['author'] == user_id]['body'].values[0][:100]}..."
    )
    print("-" * 30)
    for i, rec_id in enumerate(recs):
        text = unique_users[unique_users["author"] == rec_id]["body"].values[0][:100]
        print(f"Rec {i + 1}: {rec_id} | Text: {text}...")


# Example call
test_recommendation(user_id_list[1])

Target User (Thompson_S_Sweetback) history sample: 1. Long term economic disincentives will not be very effective because people are naturally optimist...
------------------------------
Rec 1: Jaberkaty | Text: Excellent points. I would only add that most governments are going to try to add incentive to create...
Rec 2: ScottSadistic | Text: Thats a broad statement. without minimum wage, this recession could be much worse. Not to mention al...
Rec 3: SmokingCyclist | Text: I don't disagree per se (I don't think it's a matter of survival, but I can see there would be consi...


In [26]:
print(user_id_list[1])
recommend_cnn(user_id_list[1])

Thompson_S_Sweetback


['Jaberkaty',
 'ScottSadistic',
 'SmokingCyclist',
 'inmatarian',
 'elspazzz',
 'Bossman759',
 'Yownine',
 'Twonames',
 'broca16',
 'morganshen']

In [27]:
def recommend_echo_aware(u, k=10, alpha=0.7):

    if u not in joint_embeddings:
        return []

    candidates = build_candidate_pool(u)
    existing = train_edges.get(u, set())

    # Filter valid candidates
    candidates = [
        v for v in candidates
        if v in joint_embeddings and v != u and v not in existing
    ]

    selected = []

    while len(selected) < k and candidates:
        best_v = None
        best_score = -1e9

        for v in candidates:

            # Relevance
            relevance = np.dot(joint_embeddings[u], joint_embeddings[v])

            # Redundancy (max similarity to already selected)
            redundancy = 0
            if selected:
                redundancy = max(
                    np.dot(joint_embeddings[v], joint_embeddings[r])
                    for r in selected
                )

            # MMR score
            score = alpha * relevance - (1 - alpha) * redundancy

            if score > best_score:
                best_score = score
                best_v = v

        if best_v is None:
            break

        selected.append(best_v)
        candidates.remove(best_v)

    return selected

In [28]:
def recommend_pop_minus_similarity(user, k=10, 
                                    lambda_penalty=0.5):

    if user not in joint_embeddings:
        return []

    seen = set(pos_train[pos_train["u"] == user]["v"].values)

    max_pop = max(popularity_scores.values())

    scored = []

    for candidate in global_pop_ranking:

        if candidate == user or candidate in seen:
            continue
        if candidate not in joint_embeddings:
            continue

        pop_score = popularity_scores.get(candidate, 0) / max_pop
        sim_score = np.dot(joint_embeddings[user],
                           joint_embeddings[candidate])

        score = pop_score - lambda_penalty * sim_score

        scored.append((candidate, score))

        if len(scored) >= 300:
            break

    scored.sort(key=lambda x: x[1], reverse=True)

    return [v for v, _ in scored[:k]]

# 7. Evaluation

### 7.1 Build Ground Truth

In [29]:
# 1. Build neighbor dictionaries
train_neighbors = pos_train.groupby("u")["v"].apply(set).to_dict()
test_neighbors = pos_test.groupby("u")["v"].apply(set).to_dict()

# 2. Users that exist in embedding index
embedded_users = set(user_vectors.keys())

ground_truth = {}

for u in test_neighbors:
    # Skip users without embeddings (cannot generate recommendations)
    if u not in embedded_users:
        continue

    train_set = train_neighbors.get(u, set())

    # Remove already seen interactions (only new links)
    new_interactions = test_neighbors[u] - train_set

    # Keep only targets that also have embeddings
    new_interactions = {v for v in new_interactions if v in embedded_users}

    # Only keep users with at least one evaluable target
    if len(new_interactions) > 0:
        ground_truth[u] = new_interactions


In [30]:
# Number of valid future targets per user
gt_sizes = {u: len(vs) for u, vs in ground_truth.items()}

# Convert to DataFrame for easier inspection
gt_df = pd.DataFrame.from_dict(gt_sizes, orient="index", columns=["n_targets"])
gt_df.index.name = "user"

gt_df.head()


,n_targets
user,
1r0n1k,2
25X,2
294116002,6
2xwhyzed,2
3DBeerGoggles,2


### 7.2 Evaluate Precision@K

In [31]:
def evaluate_precision_at_k(model_recommend_fn, ground_truth, k=10):
    """
    model_recommend_fn: function(user_id, k) -> list of recommended users
    ground_truth: dict {u: set(valid target users)}
    """
    total_hits = 0
    total_users = 0

    for u, true_targets in ground_truth.items():
        recs = model_recommend_fn(u, k=k)

        # Safety: ensure only evaluable users are recommended
        recs = [r for r in recs if r in user_vectors]

        hits = len(set(recs) & true_targets)

        total_hits += hits
        total_users += 1

    if total_users == 0:
        return 0.0

    precision = total_hits / (k * total_users)
    return precision

In [32]:
precision_random = evaluate_precision_at_k(recommend_random, ground_truth, k=10)
precision_popularity = evaluate_precision_at_k(recommend_popularity, ground_truth, k=10)
precision_common_neighbors = evaluate_precision_at_k(recommend_common_neighbors, ground_truth, k=10)
precision_cnn = evaluate_precision_at_k(recommend_cnn, ground_truth, k=10)
precision_pop_sim = evaluate_precision_at_k(recommend_pop_minus_similarity, ground_truth, k=10)

print("Random Baseline Precision@10:", precision_random)
print("Popularity Baseline Precision@10:", precision_popularity)
print("Common Neighbors Baseline Precision@10:", precision_common_neighbors)
print("CNN Precision@10:", precision_cnn)
print("Pop-Sim Precision@10:", precision_pop_sim)

Random Baseline Precision@10: 0.0013157894736842105
Popularity Baseline Precision@10: 0.021052631578947368
Common Neighbors Baseline Precision@10: 0.008771929824561403
CNN Precision@10: 0.005701754385964913
Pop-Sim Precision@10: 0.018859649122807017


### 7.3 Evaluate Recall@K

In [33]:
def evaluate_recall_at_k(model_recommend_fn, ground_truth, k=10):
    """
    model_recommend_fn: function(user_id, k) -> list of recommended users
    ground_truth: dict {u: set(valid target users)}
    """

    total_recall = 0.0
    total_users = 0

    for u, true_targets in ground_truth.items():
        recs = model_recommend_fn(u, k=k)

        # Safety: ensure candidate universe consistency
        recs = [r for r in recs if r in user_vectors]

        hits = len(set(recs) & true_targets)

        recall_u = hits / len(true_targets)

        total_recall += recall_u
        total_users += 1

    if total_users == 0:
        return 0.0

    return total_recall / total_users

In [34]:
recall_random = evaluate_recall_at_k(
    recommend_random,
    ground_truth,
    k=10
)

recall_popularity = evaluate_recall_at_k(
    recommend_popularity,
    ground_truth,
    k=10
)

recall_common_neighbors = evaluate_recall_at_k(
    recommend_common_neighbors,
    ground_truth,
    k=10
)

recall_cnn = evaluate_recall_at_k(
    recommend_cnn,
    ground_truth,
    k=10
)

recall_pop_sim = evaluate_recall_at_k(
    recommend_pop_minus_similarity,
    ground_truth,
    k=10
)

print("Random Baseline Recall@10:", recall_random)
print("Popularity Baseline Recall@10:", recall_popularity)
print("Common Neighbors Baseline Recall@10:", recall_common_neighbors)
print("CNN Recall@10:", recall_cnn)
print("Pop-Sim Recall@10:", recall_pop_sim)

Random Baseline Recall@10: 0.0
Popularity Baseline Recall@10: 0.10801260527714819
Common Neighbors Baseline Recall@10: 0.0361071471874796
CNN Recall@10: 0.027095516569200784
Pop-Sim Recall@10: 0.1039434044974211


### 7.4 Evaluate nDCG@K

In [35]:
def evaluate_ndcg_at_k(model_recommend_fn, ground_truth, k=10):
    """
    model_recommend_fn: function(user_id, k) -> ranked list
    ground_truth: dict {u: set(valid targets)}
    """
    
    total_ndcg = 0.0
    total_users = 0

    for u, true_targets in ground_truth.items():
        recs = model_recommend_fn(u, k=k)

        # Compute DCG
        dcg = 0.0
        for rank, candidate in enumerate(recs, start=1):
            if candidate in true_targets:
                dcg += 1.0 / np.log2(rank + 1)

        # Compute IDCG
        ideal_hits = min(len(true_targets), k)
        idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))

        if idcg == 0:
            continue  # skip users with no valid ground truth (safety)

        ndcg_u = dcg / idcg

        total_ndcg += ndcg_u
        total_users += 1

    if total_users == 0:
        return 0.0

    return total_ndcg / total_users

In [36]:
ndcg_random = evaluate_ndcg_at_k(
    recommend_random,
    ground_truth,
    k=10
)

ndcg_popularity = evaluate_ndcg_at_k(
    recommend_popularity,
    ground_truth,
    k=10
)

ndcg_common_neighbors = evaluate_ndcg_at_k(
    recommend_common_neighbors,
    ground_truth,
    k=10
)

ndcg_cnn = evaluate_ndcg_at_k(
    recommend_cnn,
    ground_truth,
    k=10
)

ndcg_pop_sim = evaluate_ndcg_at_k(
    recommend_pop_minus_similarity,
    ground_truth,
    k=10
)

print("Random Baseline nDCG@10:", ndcg_random)
print("Popularity Baseline nDCG@10:", ndcg_popularity)
print("Common Neighbors Baseline nDCG@10:", ndcg_common_neighbors)
print("CNN nDCG@10:", ndcg_cnn)
print("Pop-Sim nDCG@10:", ndcg_pop_sim)

Random Baseline nDCG@10: 0.005764872115597313
Popularity Baseline nDCG@10: 0.06940524104739212
Common Neighbors Baseline nDCG@10: 0.020901455365493662
CNN nDCG@10: 0.016011508255128294
Pop-Sim nDCG@10: 0.0633239509243952


### 7.5 Evaluate Echo Chamber Metrics

##### 7.5.1 Defining individual diversity and novelty functions

In [37]:
dm_map = {}
for user in user_id_list:
    if user in graph_embeddings and user in text_embeddings:
        # L2-Normalize each part individually to balance their influence
        g_norm = graph_embeddings[user] / (np.linalg.norm(graph_embeddings[user]) + 1e-9)
        t_norm = text_embeddings[user] / (np.linalg.norm(text_embeddings[user]) + 1e-9)
        # Result is a 192-dim unified vector
        dm_map[user] = np.concatenate([g_norm, t_norm])
        dm_map[user] = dm_map[user] / np.linalg.norm(dm_map[user])

def get_dm(u, v):
    return np.linalg.norm(dm_map[u] - dm_map[v])

train_edges = defaultdict(set)

for u, v in pos_train[["u", "v"]].itertuples(index=False):
    train_edges[u].add(v)   # only outgoing

In [38]:
def calculate_dm_distance(u, v):
    return np.linalg.norm(dm_map[u] - dm_map[v])

# Individual Diversity
def individual_diversity(recs):
    # Average dm between all pairs in the Top-10
    dists = [calculate_dm_distance(i, j) for i in recs for j in recs if i != j]
    return np.mean(dists) if dists else 0

# Individual Novelty
def individual_novelty(recs, existing_interactions):
    # Average dm between Top-10 and the Training set (Fu)
    dists = [calculate_dm_distance(r, f) for r in recs for f in existing_interactions]
    return np.mean(dists) if dists else 0

##### 7.5.2 Evaluate Individual Diversity @ k

In [39]:
def evaluate_individual_diversity_at_k(model_recommend_fn, k=10):
    total_div = 0.0
    total_users = 0

    for u in ground_truth.keys():
        recs = model_recommend_fn(u, k=k)

        # Only consider users with at least 2 recommendations
        if len(recs) < 2:
            continue

        dists = []

        for i in range(len(recs)):
            for j in range(i + 1, len(recs)):
                if recs[i] in dm_map and recs[j] in dm_map:
                    d = calculate_dm_distance(recs[i], recs[j])
                    dists.append(d)

        if len(dists) == 0:
            continue

        total_div += np.mean(dists)
        total_users += 1

    return total_div / total_users if total_users > 0 else 0.0

In [40]:
indiv_div_random = evaluate_individual_diversity_at_k(
    recommend_random,
    k=10
)

indiv_div_popularity = evaluate_individual_diversity_at_k(
    recommend_popularity,
    k=10
)

indiv_div_common_neighbors = evaluate_individual_diversity_at_k(
    recommend_common_neighbors,
    k=10
)

indiv_div_cnn = evaluate_individual_diversity_at_k(
    recommend_cnn,
    k=10
)

indiv_div_pop_sim = evaluate_individual_diversity_at_k(
    recommend_pop_minus_similarity,
    k=10
)

print("Random Baseline Individual Diversity@10:", indiv_div_random)
print("Popularity Baseline Individual Diversity@10:", indiv_div_popularity)
print("Common Neighbors Baseline Individual Diversity@10:", indiv_div_common_neighbors)
print("CNN Individual Diversity@10:", indiv_div_cnn)
print("Pop-Sim Individual Diversity@10:", indiv_div_pop_sim)

Random Baseline Individual Diversity@10: 1.3610716650360508
Popularity Baseline Individual Diversity@10: 1.3311586589143987
Common Neighbors Baseline Individual Diversity@10: 1.3018726194482544
CNN Individual Diversity@10: 1.2337075105884618
Pop-Sim Individual Diversity@10: 1.330899901557387


##### 7.5.3 Evaluate Individual Novelty @ k

In [41]:
def evaluate_individual_novelty_at_k(model_recommend_fn, k=10):
    total_novelty = 0.0
    total_users = 0

    for u in ground_truth.keys():
        recs = model_recommend_fn(u, k=k)

        # Training neighbors (historical interactions)
        F_u = train_edges.get(u, set())

        # Need at least 1 recommendation and 1 historical neighbor
        if len(recs) == 0 or len(F_u) == 0:
            continue

        dists = []

        for r in recs:
            if r not in dm_map:
                continue
            for f in F_u:
                if f not in dm_map:
                    continue
                d = calculate_dm_distance(r, f)
                dists.append(d)

        if len(dists) == 0:
            continue

        total_novelty += np.mean(dists)
        total_users += 1

    return total_novelty / total_users if total_users > 0 else 0.0

In [42]:
indiv_nov_random = evaluate_individual_novelty_at_k(
    recommend_random,
    k=10
)

indiv_nov_popularity = evaluate_individual_novelty_at_k(
    recommend_popularity,
    k=10
)

indiv_nov_common_neighbors = evaluate_individual_novelty_at_k(
    recommend_common_neighbors,
    k=10
)

indiv_nov_cnn = evaluate_individual_novelty_at_k(
    recommend_cnn,
    k=10
)

indiv_nov_pop_sim = evaluate_individual_novelty_at_k(
    recommend_pop_minus_similarity,
    k=10
)

print("Random Baseline Individual Novelty@10:", indiv_nov_random)
print("Popularity Baseline Individual Novelty@10:", indiv_nov_popularity)
print("Common Neighbors Baseline Individual Novelty@10:", indiv_nov_common_neighbors)
print("Pop-Sim Individual Novelty@10:", indiv_nov_pop_sim)

Random Baseline Individual Novelty@10: 1.3709799104982667
Popularity Baseline Individual Novelty@10: 1.3584340107333552
Common Neighbors Baseline Individual Novelty@10: 1.294432843432707
Pop-Sim Individual Novelty@10: 1.3653171857198079


##### 7.5.4 Get communities with louvain to evaluate community based metrics

In [43]:
# 1️⃣ Build UNDIRECTED weighted graph
G = nx.Graph()

# Count interaction frequency (edge weights)
edge_weights = (
    pos_train.groupby(["u", "v"])
    .size()
    .reset_index(name="weight")
)

for u, v, w in edge_weights.itertuples(index=False):
    if G.has_edge(u, v):
        G[u][v]["weight"] += w
    else:
        G.add_edge(u, v, weight=w)

# Detect communities on training graph
partition = community_louvain.best_partition(G)

# partition: dict {user -> community_id}
community_members = defaultdict(set)

for user, comm in partition.items():
    community_members[comm].add(user)


##### 7.5.5 Evaluate Community Diversity @ k 

In [44]:
def community_diversity(recommend_fn, k=10):
    community_scores = []

    for comm, members in community_members.items():
        R_c = set()

        for u in members:
            if u in user_vectors:
                R_c.update(recommend_fn(u, k=k))

        R_c = list(R_c)

        if len(R_c) < 2:
            continue

        dists = [
            calculate_dm_distance(i, j)
            for i in R_c for j in R_c
            if i != j
        ]

        if dists:
            community_scores.append(np.mean(dists))

    return np.mean(community_scores) if community_scores else 0.0

In [45]:
comm_div_random = community_diversity(
    recommend_random,
    k=10
)

comm_div_popularity = community_diversity(
    recommend_popularity,
    k=10
)

comm_div_common_neighbors = community_diversity(
    recommend_common_neighbors,
    k=10
)

comm_div_cnn = community_diversity(
    recommend_cnn,
    k=10
)


comm_div_pop_sim = community_diversity(
    recommend_pop_minus_similarity,
    k=10
)

print("Random Baseline Community Diversity@10:", comm_div_random)
print("Popularity Baseline Community Diversity@10:", comm_div_popularity)
print("Common Neighbors Community Diversity@10:", comm_div_common_neighbors)
print("CNN Community Diversity@10:", comm_div_cnn)
print("Pop-Sim Community Diversity@10:", comm_div_pop_sim)

Random Baseline Community Diversity@10: 1.3616719
Popularity Baseline Community Diversity@10: 1.330313
Common Neighbors Community Diversity@10: 1.2654669
CNN Community Diversity@10: 1.2601428
Pop-Sim Community Diversity@10: 1.3326603


##### 7.5.6 Evaluate Community Novelty @ k

In [46]:
def community_novelty(recommend_fn, k=10):
    community_scores = []

    for comm, members in community_members.items():

        R_c = set()
        F_c = set()

        for u in members:
            if u in user_vectors:
                R_c.update(recommend_fn(u, k=k))
                F_c.update(train_edges.get(u, set()))

        R_c = list(R_c)
        F_c = list(F_c)

        if len(R_c) == 0 or len(F_c) == 0:
            continue

        dists = [
            calculate_dm_distance(i, j)
            for i in R_c for j in F_c
        ]

        if dists:
            community_scores.append(np.mean(dists))

    return np.mean(community_scores) if community_scores else 0.0

In [47]:
comm_nov_random = community_novelty(
    recommend_random,
    k=10
)

comm_nov_popularity = community_novelty(
    recommend_popularity,
    k=10
)

comm_nov_common_neighbors = community_novelty(
    recommend_common_neighbors,
    k=10
)

comm_nov_cnn = community_novelty(
    recommend_cnn,
    k=10
)

comm_nov_pop_sim = community_novelty(
    recommend_pop_minus_similarity,
    k=10
)

print("Random Baseline Community Novelty@10:", comm_nov_random)
print("Popularity Baseline Community Novelty@10:", comm_nov_popularity)
print("Common Neighbors Community Novelty@10:", comm_nov_common_neighbors)
print("CNN Community Novelty@10:", comm_nov_cnn)
print("Pop-Sim Community Novelty@10:", comm_nov_pop_sim)

Random Baseline Community Novelty@10: 1.3490942
Popularity Baseline Community Novelty@10: 1.38938
Common Neighbors Community Novelty@10: 1.2193974
CNN Community Novelty@10: 1.229522
Pop-Sim Community Novelty@10: 1.3973573
